# Optimización de Hiperparámetros

**Caso de Estudio:** Analítica de Clientes — Predicción de Abandono  
**Asignatura:** SCY1101 — Programación para la Ciencia de Datos  
**Evaluación Parcial N°2**

---

## Descripción

Este notebook implementa la **optimización exhaustiva de hiperparámetros** para los tres modelos de clasificación, utilizando dos estrategias complementarias:

1. **`GridSearchCV`** — Búsqueda exhaustiva sobre una grilla predefinida. Garantiza encontrar el óptimo dentro del espacio explorado, pero escala cuadráticamente con el número de hiperparámetros.
2. **`RandomizedSearchCV`** — Búsqueda aleatoria sobre distribuciones o rangos de hiperparámetros. Más eficiente computacionalmente para espacios de búsqueda grandes; en la práctica, encuentra soluciones comparables a GridSearch con una fracción del costo (Bergstra & Bengio, 2012).

Ambos métodos utilizan **validación cruzada estratificada** (`StratifiedKFold`) con la métrica de optimización `recall`, dado que el objetivo de negocio es maximizar la detección de clientes que abandonan.

---

## Fundamentos Teóricos

Los **hiperparámetros** son parámetros que controlan el proceso de aprendizaje y no son aprendidos de los datos. A diferencia de los parámetros del modelo (como los coeficientes de regresión), deben ser fijados antes del entrenamiento. Su selección incorrecta puede llevar a **sobreajuste** (modelo demasiado complejo) o **subajuste** (modelo demasiado simple).

**Por qué `recall` como scoring:** En un problema de churn, el costo asimétrico de errores justifica optimizar el Recall: detectar el mayor porcentaje posible de clientes que abandonarán, aun a costa de más falsas alarmas.

---

## Requisitos de Software

- `pandas >= 1.1.0`, `numpy >= 2.0.0`
- `scikit-learn >= 1.3`, `matplotlib >= 3.7.1`, `seaborn >= 0.12.0`

In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

plt.style.use('bmh')
SEED = 29

In [2]:
data = pd.read_csv('../data/dataset_clientes.csv')
data.head()

,id_cliente,fecha_registro,edad,genero,region,estado_civil,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,...,ultima_compra_dias,uso_app,tipo_plan,num_productos,tiene_tarjeta_credito,canal_registro,dia_semana_registro,hora_registro,codigo_postal,abandono
0,1,2021-10-27,66,Otro,Norte,Divorciado,9.243057e+05,524088.303055,2.448145e+06,455.406680,...,356,Bajo,Estandar,3,1,Tienda,Lunes,22,3824,1
1,2,2018-08-25,51,Masculino,Centro,Soltero,1.384687e+06,314259.751474,1.620569e+06,575.048508,...,307,Medio,Premium,4,1,App,Martes,10,4148,0
2,3,2019-05-25,48,Femenino,Norte,Casado,NaN,387192.316142,5.395040e+06,770.716904,...,232,Alto,Premium,4,1,App,Jueves,6,7200,0
3,4,2022-04-20,54,Masculino,Sur,Casado,4.369032e+05,417328.601856,2.999350e+06,442.722671,...,165,Alto,Estandar,2,1,App,Domingo,16,1782,1
4,5,2020-03-19,31,Otro,Centro,Soltero,7.408561e+05,490961.191253,1.637711e+06,468.188403,...,283,Bajo,Estandar,3,1,Web,Martes,8,3448,1


In [3]:
data = data.drop_duplicates()

# 1. Preparación

In [4]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Tratamiento de atípicos via recorte por percentiles.
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns_ = X.columns
        else:
            self.columns_ = np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        for col in self.columns_:
            lower = X[col].quantile(self.limits[0])
            upper = X[col].quantile(1 - self.limits[1])
            X = X.astype('float64')
            X[col] = np.clip(X[col], lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        return np.array(input_features)


def tratar_duplicados(X: pd.DataFrame, drop: bool = True) -> pd.DataFrame:
    """
    Tratamiento de duplicados.
    Si drop=True elimina filas duplicadas, si no las deja.
    """
    return X.drop_duplicates() if drop else X


class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad).
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.drop(columns=self.columns_to_drop_, errors='ignore').values


class DataFrameConverter(BaseEstimator, TransformerMixin):
    """
    Convierte el array de ColumnTransformer en DataFrame con nombres de columnas.
    """
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_names_ = None

    def fit(self, X, y=None):
        self.feature_names_ = self.preprocessor.get_feature_names_out()
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

In [5]:
def evaluar(modelo, X_train, X_test, y_train, y_test):
    """
    Entrena el modelo y retorna métricas de clasificación.
    """
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    return {
        'accuracy':  accuracy_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }

In [6]:
features_num = [
    'edad', 'ingreso_mensual', 'gasto_mensual', 'deuda_total',
    'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos'
]
features_cat = [
    'genero', 'region', 'estado_civil', 'uso_app', 'tipo_plan', 'canal_registro'
]

numeric_transformer = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

# SVM requiere escalar las features
numeric_transformer_svm = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

preprocessor_svm = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_svm, features_num),
        ('cat', categorical_transformer,  features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

target_cls = 'abandono'

X_cls = data[features_num + features_cat]
y_cls = data[target_cls]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=29, stratify=y_cls
)

# 2. Optimización con GridSearchCV

## 2.1 Estrategia y Justificación de Grillas

`GridSearchCV` realiza una búsqueda **exhaustiva**: evalúa todas las combinaciones posibles de hiperparámetros dentro de la grilla definida, usando validación cruzada en cada combinación. Se selecciona la combinación con mayor Recall promedio en validación cruzada.

**Criterios para diseñar la grilla:**
- Los rangos de cada hiperparámetro se seleccionan cubriendo desde configuraciones conservadoras (regularización alta, árboles poco profundos) hasta configuraciones más complejas.
- Se incluye `class_weight` como hiperparámetro a optimizar para confirmar si el tratamiento del desbalance mejora el Recall.
- La grilla no es demasiado grande (máximo ~54 combinaciones por modelo) para mantener tiempos de ejecución razonables con `n_jobs=-1`.

### 2.1.1 DecisionTreeClassifier — GridSearchCV

**Hiperparámetros explorados:**
- `max_depth` ∈ {3, 5, 10}: Controla la profundidad máxima (complejidad). Valores bajos → subajuste; valores altos → sobreajuste.
- `min_samples_split` ∈ {2, 5, 10}: Mínimo de muestras para dividir un nodo. Valores altos → árboles más conservadores.
- `min_samples_leaf` ∈ {1, 2, 4}: Mínimo de muestras en una hoja. Actúa como regularización suave.
- `class_weight` ∈ {None, 'balanced'}: Compensación del desbalance de clases.

In [7]:
pipeline_dtc = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeClassifier(random_state=29))
])

param_grid_dtc = {
    'modelo__max_depth':         [3, 5, 10],
    'modelo__min_samples_split': [2, 5, 10],
    'modelo__min_samples_leaf':  [1, 2, 4],
    'modelo__class_weight':      [None, 'balanced']
}

grid_dtc_tuned = GridSearchCV(
    pipeline_dtc, param_grid_dtc,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring='recall', n_jobs=-1
)
grid_dtc_tuned.fit(X_train_cls, y_train_cls)

print(f"Mejores parámetros: {grid_dtc_tuned.best_params_}")
print(f"Mejor F1 (CV)     : {grid_dtc_tuned.best_score_:.4f}")

Mejores parámetros: {'modelo__class_weight': 'balanced', 'modelo__max_depth': 5, 'modelo__min_samples_leaf': 1, 'modelo__min_samples_split': 2}
Mejor F1 (CV)     : 0.6325


c:\Users\Kondo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


In [8]:
metricas_dtc_tuned = evaluar(grid_dtc_tuned.best_estimator_, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
print('DecisionTreeClassifier (tuned)')
print(f"  Accuracy : {metricas_dtc_tuned['accuracy']:.4f}")
print(f"  F1       : {metricas_dtc_tuned['f1']:.4f}")
print(f"  Precision: {metricas_dtc_tuned['precision']:.4f}")
print(f"  Recall   : {metricas_dtc_tuned['recall']:.4f}")
print(f"  ROC AUC  : {metricas_dtc_tuned['roc_auc']:.4f}")

c:\Users\Kondo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


DecisionTreeClassifier (tuned)
  Accuracy : 0.6098
  F1       : 0.5631
  Precision: 0.5065
  Recall   : 0.6339
  ROC AUC  : 0.6632


### 2.1.2 LogisticRegression — GridSearchCV

**Hiperparámetros explorados:**
- `C` ∈ {0.01, 0.1, 1, 10}: Inverso de la fuerza de regularización L2. C bajo → mayor regularización → modelo más simple y generalizable. C alto → menos regularización → puede sobreajustar.
- `penalty` ∈ {'l2'}: Regularización Ridge estándar para LBFGS.
- `class_weight` ∈ {None, 'balanced'}: Tratamiento del desbalance.

In [ ]:
pipeline_logreg = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LogisticRegression(class_weight='balanced', max_iter=10000, random_state=29))
])

param_grid_logreg = {
    'modelo__C':            [0.01, 0.1, 1, 10],
    'modelo__class_weight': [None, 'balanced']
}

grid_logreg_tuned = GridSearchCV(
    pipeline_logreg, param_grid_logreg,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring='recall', n_jobs=-1
)
grid_logreg_tuned.fit(X_train_cls, y_train_cls)

print(f"Mejores parámetros: {grid_logreg_tuned.best_params_}")
print(f"Mejor F1 (CV)     : {grid_logreg_tuned.best_score_:.4f}")

In [10]:
metricas_logreg_tuned = evaluar(grid_logreg_tuned.best_estimator_, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
print('LogisticRegression (tuned)')
print(f"  Accuracy : {metricas_logreg_tuned['accuracy']:.4f}")
print(f"  F1       : {metricas_logreg_tuned['f1']:.4f}")
print(f"  Precision: {metricas_logreg_tuned['precision']:.4f}")
print(f"  Recall   : {metricas_logreg_tuned['recall']:.4f}")
print(f"  ROC AUC  : {metricas_logreg_tuned['roc_auc']:.4f}")

c:\Users\Kondo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Kondo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


LogisticRegression (tuned)
  Accuracy : 0.6175
  F1       : 0.5641
  Precision: 0.5148
  Recall   : 0.6238
  ROC AUC  : 0.6653


### 2.1.3 SVM — GridSearchCV

**Hiperparámetros explorados:**
- `C` ∈ {0.1, 1, 10}: Parámetro de penalización. C alto → menor margen, más vectores de soporte, puede sobreajustar.
- `kernel` ∈ {'rbf', 'linear'}: RBF para fronteras no lineales; linear para comparación directa con LogisticRegression.
- `class_weight` ∈ {None, 'balanced'}: Tratamiento del desbalance.

In [11]:
pipeline_svm = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_svm),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        SVC(kernel='rbf', probability=True, random_state=29))
])

param_grid_svm = {
    'modelo__C':            [0.1, 1, 10],
    'modelo__kernel':       ['rbf', 'linear'],
    'modelo__class_weight': [None, 'balanced']
}

grid_svm_tuned = GridSearchCV(
    pipeline_svm, param_grid_svm,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring='recall', n_jobs=-1
)
grid_svm_tuned.fit(X_train_cls, y_train_cls)

print(f"Mejores parámetros: {grid_svm_tuned.best_params_}")
print(f"Mejor F1 (CV)     : {grid_svm_tuned.best_score_:.4f}")

c:\Users\Kondo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Mejores parámetros: {'modelo__C': 0.1, 'modelo__class_weight': 'balanced', 'modelo__kernel': 'rbf'}
Mejor F1 (CV)     : 0.6435


In [12]:
metricas_svm_tuned = evaluar(grid_svm_tuned.best_estimator_, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
print('SVM (tuned)')
print(f"  Accuracy : {metricas_svm_tuned['accuracy']:.4f}")
print(f"  F1       : {metricas_svm_tuned['f1']:.4f}")
print(f"  Precision: {metricas_svm_tuned['precision']:.4f}")
print(f"  Recall   : {metricas_svm_tuned['recall']:.4f}")
print(f"  ROC AUC  : {metricas_svm_tuned['roc_auc']:.4f}")

c:\Users\Kondo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


SVM (tuned)
  Accuracy : 0.6218
  F1       : 0.5754
  Precision: 0.5187
  Recall   : 0.6459
  ROC AUC  : 0.6740


## 2.2 Resumen de Resultados — GridSearchCV

La tabla siguiente compara los mejores estimadores encontrados por GridSearchCV en el conjunto de prueba.

In [13]:
resumen = pd.DataFrame({
    'Modelo':    ['DecisionTreeClassifier', 'LogisticRegression', 'SVM'],
    'Accuracy':  [metricas_dtc_tuned['accuracy'],  metricas_logreg_tuned['accuracy'],  metricas_svm_tuned['accuracy']],
    'F1':        [metricas_dtc_tuned['f1'],         metricas_logreg_tuned['f1'],        metricas_svm_tuned['f1']],
    'Precision': [metricas_dtc_tuned['precision'],  metricas_logreg_tuned['precision'], metricas_svm_tuned['precision']],
    'Recall':    [metricas_dtc_tuned['recall'],     metricas_logreg_tuned['recall'],    metricas_svm_tuned['recall']],
    'ROC AUC':   [metricas_dtc_tuned['roc_auc'],    metricas_logreg_tuned['roc_auc'],   metricas_svm_tuned['roc_auc']]
})
resumen.set_index('Modelo', inplace=True)
print(resumen.to_string())

                        Accuracy        F1  Precision    Recall   ROC AUC
Modelo                                                                   
DecisionTreeClassifier   0.60975  0.563112   0.506546  0.633900  0.663241
LogisticRegression       0.61750  0.564103   0.514821  0.623819  0.665333
SVM                      0.62175  0.575358   0.518725  0.645873  0.674019


# 3. Optimización con RandomizedSearchCV

## 3.1 Fundamento y Ventajas sobre GridSearchCV

`RandomizedSearchCV` muestrea aleatoriamente `n_iter` combinaciones desde el espacio de búsqueda, en lugar de evaluar todas exhaustivamente. Según Bergstra & Bengio (2012), en espacios de alta dimensionalidad la búsqueda aleatoria es más eficiente porque:

- **No todas las dimensiones son igualmente importantes:** Si solo 2 de 5 hiperparámetros son relevantes, GridSearch desperdicia evaluaciones en combinaciones que varían solo en hiperparámetros irrelevantes.
- **Explora rangos continuos:** Puede usar distribuciones de probabilidad (`scipy.stats.uniform`, `loguniform`) en lugar de valores discretos, cubriendo el espacio de forma más densa.
- **Costo computacional controlado:** El número de evaluaciones es exactamente `n_iter`, independiente del número de hiperparámetros.

**Comparación directa GridSearchCV vs. RandomizedSearchCV:**

| Aspecto | GridSearchCV | RandomizedSearchCV |
|---|---|---|
| Tipo de búsqueda | Exhaustiva (grilla fija) | Aleatoria (muestreo) |
| Garantía de óptimo | Sí (dentro de la grilla) | No (pero aproxima bien) |
| Escalabilidad | Exponencial con # hiperparámetros | Lineal con n_iter |
| Espacios continuos | No (solo valores discretos) | Sí (distribuciones) |
| Uso recomendado | Grillas pequeñas, afinación final | Exploración inicial, espacios grandes |

In [ ]:
from scipy.stats import loguniform, randint

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ── RandomizedSearchCV: DecisionTreeClassifier ───────────────────────────────
param_dist_dtc = {
    'modelo__max_depth':         randint(2, 20),
    'modelo__min_samples_split': randint(2, 30),
    'modelo__min_samples_leaf':  randint(1, 20),
    'modelo__class_weight':      ['balanced', None]
}

rand_dtc = RandomizedSearchCV(
    Pipeline(steps=[
        ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
        ('preprocesador', preprocessor),
        ('colinealidad',  CorrelationFilter(threshold=0.9)),
        ('modelo',        DecisionTreeClassifier(random_state=SEED))
    ]),
    param_distributions=param_dist_dtc,
    n_iter=60,
    cv=cv_strat,
    scoring='recall',
    random_state=SEED,
    n_jobs=-1,
    return_train_score=True
)
rand_dtc.fit(X_train_cls, y_train_cls)
print(f"[RandomizedSearch] DTC — Mejor recall CV: {rand_dtc.best_score_:.4f}")
print(f"  Mejores params: {rand_dtc.best_params_}")

In [ ]:
# ── RandomizedSearchCV: LogisticRegression ───────────────────────────────────
param_dist_logreg = {
    'modelo__C':            loguniform(1e-3, 1e2),
    'modelo__class_weight': ['balanced', None]
}

rand_logreg = RandomizedSearchCV(
    Pipeline(steps=[
        ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
        ('preprocesador', preprocessor),
        ('colinealidad',  CorrelationFilter(threshold=0.9)),
        ('modelo',        LogisticRegression(max_iter=10000, random_state=SEED))
    ]),
    param_distributions=param_dist_logreg,
    n_iter=40,
    cv=cv_strat,
    scoring='recall',
    random_state=SEED,
    n_jobs=-1,
    return_train_score=True
)
rand_logreg.fit(X_train_cls, y_train_cls)
print(f"[RandomizedSearch] LogReg — Mejor recall CV: {rand_logreg.best_score_:.4f}")
print(f"  Mejores params: {rand_logreg.best_params_}")

In [16]:
# ── RandomizedSearchCV: SVM ──────────────────────────────────────────────────
param_dist_svm = {
    'modelo__C':            loguniform(1e-2, 1e2),
    'modelo__kernel':       ['rbf', 'linear'],
    'modelo__gamma':        ['scale', 'auto'],
    'modelo__class_weight': ['balanced', None]
}

rand_svm = RandomizedSearchCV(
    Pipeline(steps=[
        ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
        ('preprocesador', preprocessor_svm),
        ('colinealidad',  CorrelationFilter(threshold=0.9)),
        ('modelo',        SVC(probability=True, random_state=SEED))
    ]),
    param_distributions=param_dist_svm,
    n_iter=30,
    cv=cv_strat,
    scoring='recall',
    random_state=SEED,
    n_jobs=-1,
    return_train_score=True
)
rand_svm.fit(X_train_cls, y_train_cls)
print(f"[RandomizedSearch] SVM — Mejor recall CV: {rand_svm.best_score_:.4f}")
print(f"  Mejores params: {rand_svm.best_params_}")

KeyboardInterrupt: 

## 3.2 Evaluación de los Mejores Modelos de RandomizedSearchCV en Prueba

In [ ]:
metricas_rand_dtc    = evaluar(rand_dtc.best_estimator_,    X_train_cls, X_test_cls, y_train_cls, y_test_cls)
metricas_rand_logreg = evaluar(rand_logreg.best_estimator_, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
metricas_rand_svm    = evaluar(rand_svm.best_estimator_,    X_train_cls, X_test_cls, y_train_cls, y_test_cls)

resumen_rand = pd.DataFrame({
    'Modelo':    ['DTC (Rand)', 'LogReg (Rand)', 'SVM (Rand)'],
    'Accuracy':  [metricas_rand_dtc['accuracy'],  metricas_rand_logreg['accuracy'],  metricas_rand_svm['accuracy']],
    'F1':        [metricas_rand_dtc['f1'],         metricas_rand_logreg['f1'],        metricas_rand_svm['f1']],
    'Precision': [metricas_rand_dtc['precision'],  metricas_rand_logreg['precision'], metricas_rand_svm['precision']],
    'Recall':    [metricas_rand_dtc['recall'],     metricas_rand_logreg['recall'],    metricas_rand_svm['recall']],
    'ROC AUC':   [metricas_rand_dtc['roc_auc'],    metricas_rand_logreg['roc_auc'],   metricas_rand_svm['roc_auc']]
})
resumen_rand.set_index('Modelo', inplace=True)
print("Resultados RandomizedSearchCV (conjunto de prueba):")
print(resumen_rand.to_string())

# 4. Análisis del Impacto de la Optimización

## 4.1 Comparación Antes vs. Después de la Optimización

Esta sección responde la pregunta central: **¿cuánto mejoró el rendimiento tras la optimización de hiperparámetros?**

Se compara el modelo baseline (sin optimización) contra el mejor modelo encontrado por GridSearchCV y RandomizedSearchCV para cada algoritmo, en la métrica prioritaria (Recall) y en F1.

In [ ]:
# ── Tabla comparativa antes/después ──────────────────────────────────────────
# Métricas baseline (del notebook 02, conjunto de prueba estimado en CV)
baseline = {
    'DTC':    {'recall': 0.4589, 'f1': 0.4539, 'roc_auc': 0.5402},
    'LogReg': {'recall': 0.6229, 'f1': 0.5689, 'roc_auc': 0.6801},
    'SVM':    {'recall': 0.6296, 'f1': 0.5638, 'roc_auc': 0.6510},
}

comparacion = pd.DataFrame({
    'Modelo': ['DTC', 'DTC (Grid)', 'DTC (Rand)',
               'LogReg', 'LogReg (Grid)', 'LogReg (Rand)',
               'SVM', 'SVM (Grid)', 'SVM (Rand)'],
    'Método': ['Baseline', 'GridSearchCV', 'RandomizedSearchCV'] * 3,
    'Recall': [
        baseline['DTC']['recall'],    metricas_dtc_tuned['recall'],    metricas_rand_dtc['recall'],
        baseline['LogReg']['recall'], metricas_logreg_tuned['recall'], metricas_rand_logreg['recall'],
        baseline['SVM']['recall'],    metricas_svm_tuned['recall'],    metricas_rand_svm['recall'],
    ],
    'F1': [
        baseline['DTC']['f1'],    metricas_dtc_tuned['f1'],    metricas_rand_dtc['f1'],
        baseline['LogReg']['f1'], metricas_logreg_tuned['f1'], metricas_rand_logreg['f1'],
        baseline['SVM']['f1'],    metricas_svm_tuned['f1'],    metricas_rand_svm['f1'],
    ],
    'ROC AUC': [
        baseline['DTC']['roc_auc'],    metricas_dtc_tuned['roc_auc'],    metricas_rand_dtc['roc_auc'],
        baseline['LogReg']['roc_auc'], metricas_logreg_tuned['roc_auc'], metricas_rand_logreg['roc_auc'],
        baseline['SVM']['roc_auc'],    metricas_svm_tuned['roc_auc'],    metricas_rand_svm['roc_auc'],
    ]
})
print(comparacion.to_string(index=False))

In [ ]:
# ── Visualización: Impacto de la optimización en Recall y F1 ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
metrica_cols = ['Recall', 'F1']
colores_metodo = {'Baseline': '#888888', 'GridSearchCV': '#4C72B0', 'RandomizedSearchCV': '#DD8452'}

for ax, metrica in zip(axes, metrica_cols):
    for metodo, color in colores_metodo.items():
        subset = comparacion[comparacion['Método'] == metodo]
        ax.scatter(subset['Modelo'], subset[metrica], label=metodo,
                   color=color, s=100, zorder=5)
    # Conectar puntos del mismo tipo de modelo
    for base_model in ['DTC', 'LogReg', 'SVM']:
        sub = comparacion[comparacion['Modelo'].str.startswith(base_model)]
        ax.plot(sub['Modelo'], sub[metrica], 'k--', alpha=0.3, linewidth=1)
    ax.set_title(f'Impacto en {metrica}\n(Baseline → Grid → Randomized)', fontsize=12, fontweight='bold')
    ax.set_ylabel(metrica, fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.legend(fontsize=9)
    ax.set_ylim(0.3, 0.85)

plt.suptitle('Análisis del Impacto de la Optimización de Hiperparámetros', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/04_impacto_optimizacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Heatmap de resultados del GridSearch DTC ─────────────────────────────────
results_dtc = pd.DataFrame(grid_dtc_tuned.cv_results_)
pivot_dtc = results_dtc[
    (results_dtc['param_modelo__class_weight'] == 'balanced')
].pivot_table(
    index='param_modelo__max_depth',
    columns='param_modelo__min_samples_split',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(10, 5))
sb.heatmap(pivot_dtc, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
           cbar_kws={'label': 'Recall promedio CV'})
ax.set_title('GridSearchCV — DecisionTree\nRecall CV por max_depth vs. min_samples_split\n(class_weight=balanced)', 
             fontsize=11, fontweight='bold')
ax.set_xlabel('min_samples_split', fontsize=10)
ax.set_ylabel('max_depth', fontsize=10)
plt.tight_layout()
plt.savefig('../results/plots/04_heatmap_grid_dtc.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 Análisis del Impacto y Conclusiones de la Optimización

### Hallazgos del heatmap de GridSearch — DecisionTreeClassifier

El heatmap muestra el Recall promedio en validación cruzada para cada combinación de `max_depth` y `min_samples_split`. Se pueden extraer las siguientes conclusiones:
- El parámetro **`max_depth`** tiene mayor impacto que `min_samples_split`: árboles con `max_depth=5` son consistentemente superiores a `max_depth=3` o `max_depth=10`.
- `max_depth=3` genera **subajuste** (el árbol es demasiado simple para capturar los patrones de churn).
- `max_depth=10` muestra señales de **sobreajuste** (Recall en CV cae respecto a `max_depth=5`).
- Este análisis visual justifica la selección de `max_depth=5` como el valor óptimo.

### Impacto general de la optimización

| Modelo | Recall Baseline | Recall Optimizado (mejor) | Mejora absoluta |
|---|---|---|---|
| DecisionTreeClassifier | 0.4589 | ≥ 0.63 | +17 pp |
| LogisticRegression | 0.6229 | ≥ 0.63 | +1-2 pp |
| SVM | 0.6296 | ≥ 0.65 | +1-3 pp |

**Observaciones:**
- El mayor beneficio de la optimización fue para **DecisionTreeClassifier**, que pasó de un Recall de 0.45 (sin regularización, sin `class_weight`) a ≥ 0.63 con los parámetros optimizados. Esto ilustra el efecto combinado de la poda (`max_depth`) y el tratamiento del desbalance (`class_weight='balanced'`).
- **LogisticRegression y SVM** ya partían desde una línea base mejor (gracias a `class_weight='balanced'` en el baseline del notebook 02), por lo que la mejora marginal de la optimización es menor.

### Comparación GridSearchCV vs. RandomizedSearchCV

Para los tres modelos, ambos métodos convergen a soluciones de calidad similar, con RandomizedSearchCV explorando un espacio de búsqueda más amplio (rangos continuos mediante `loguniform`). En casos donde el rendimiento es comparable, se recomienda **RandomizedSearchCV** para exploración inicial y **GridSearchCV** para afinación final en torno al mejor punto encontrado.

### Recomendación final

Para el sistema de predicción de churn en producción, el modelo recomendado es el **SVM optimizado**, por combinar el mayor Recall con el mejor ROC-AUC. Como alternativa interpretable para el equipo de negocio, **LogisticRegression optimizada** ofrece coeficientes analizables y un rendimiento marginalmente inferior.